# ML Zoomcamp 2026 — Homework 3

Official: `materials/homework.md` (do not edit).
Data: `data/course_lead_scoring_2026.csv` (2026 pinned lead-scoring release).
Write-up: `ML_03_HW.md`.
Submit: https://courses.datatalks.club/ml-zoomcamp-2026/homework/hw03

Kernel: **ML Zoomcamp** / this folder `.venv`. Do not use `!pip` / `!wget`.


## 0) Imports


In [8]:
import os
# 운영체제(OS) 환경 변수를 설정하기 위해 파이썬 내장 os 모듈을 임포트합니다.

os.environ.setdefault(r"MPLCONFIGDIR", r"E:\IT_SPACES\AI\.cache\matplotlib")
# 맷플롯립(Matplotlib) 그래프 설정 파일이 저장될 캐시 경로를 지정하여 권한 오류나 충돌을 방지합니다.

os.environ.setdefault(r"XDG_CACHE_HOME", r"E:\IT_SPACES\AI\.cache")
# 리눅스/윈도우 환경에서 공통으로 쓰이는 XDG 캐시 기본 디렉토리를 설정합니다.

os.environ.setdefault(r"TEMP", r"E:\IT_SPACES\AI\.cache\tmp")
# 시스템 임시 파일 저장 경로(TEMP)를 특정 폴더로 고정하여 파일 접근 권한 문제를 예방합니다.

os.environ.setdefault(r"TMP", r"E:\IT_SPACES\AI\.cache\tmp")
# 대체 임시 파일 경로(TMP)도 동일한 안전 폴더로 지정합니다.

os.environ.setdefault("OMP_NUM_THREADS", "1")
# OpenMP 병렬 처리 스레드 수를 1개로 제한하여 불필요한 CPU 자원 충돌과 경고를 막습니다.

os.environ.setdefault("MKL_NUM_THREADS", "1")
# 인텔 MKL 수치 연산 라이브러리의 스레드 수를 1개로 고정하여 연산 일관성을 유지합니다.

os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
# OpenBLAS 선형대수 연산 라이브러리의 스레드 수를 1개로 제한합니다.

os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")
# NumExpr 배열 연산 스레드 수도 1개로 제한하여 다중 스레드 충돌을 완전히 방지합니다.

import numpy as np
# 다차원 배열과 고속 수학/행렬 연산을 수행하는 핵심 라이브러리인 numpy를 np라는 이름으로 불러옵니다.

import pandas as pd
# 엑셀 표 형태의 데이터를 다루고 분석하기 위한 판다스(Pandas) 라이브러리를 pd라는 이름으로 불러옵니다.

from sklearn.model_selection import train_test_split
# 전체 데이터를 학습용과 검증/테스트용 데이터셋으로 무작위 분할해 주는 사이킷런 함수를 가져옵니다.

from sklearn.metrics import mutual_info_score, accuracy_score
# 변수 간의 연관성을 측정하는 상호 정보량(mutual_info_score)과 모델 정확도를 측정하는 함수를 가져옵니다.

from sklearn.linear_model import LogisticRegression
# 이진 분류 문제를 풀기 위해 사용하는 로지스틱 회귀 모델 클래스를 가져옵니다.

## 1) Load data


In [2]:
df = pd.read_csv("data/course_lead_scoring_2026.csv")
# 지정된 경로에 있는 CSV 파일을 읽어와 판다스 DataFrame(df) 객체 형태로 메모리에 적재합니다.

df.head()
# 데이터프레임의 상단 5줄을 미리 보기로 출력하여 컬럼명과 데이터가 깨지지 않고 잘 들어왔는지 눈으로 확인합니다.


,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


## Data preparation

If missing values are present: categorical → `'NA'`, numerical → `0.0`.


In [3]:
df.isnull().sum()
# 각 컬럼별로 비어 있는 데이터(결측치, NaN)가 몇 개씩 있는지 합산하여 데이터의 결손 상태를 파악합니다.

lead_source                 148
industry                    240
employment_status           194
location                    208
annual_income               369
number_of_courses_viewed      0
interaction_count             0
lead_score                   35
converted                     0
dtype: int64

In [ ]:
# Fill missing values here (categorical 'NA', numerical 0.0).
categorical_cols = df.select_dtypes(include=['object', 'string']).columns
# .select_dtypes(include=['object', 'string'])은 지정한 문자열/범주형 타입의 컬럼들만 골라내고, .columns는 그 이름표만 추출합니다.
# -> 수많은 컬럼 중 글자 데이터만 빠짐없이 타겟팅하여 수동 입력 실수를 방지하기 위해 사용합니다.

df[categorical_cols] = df[categorical_cols].fillna('NA')
# .fillna('NA')는 선택된 컬럼들의 비어 있는 빈칸(NaN)을 'NA' 문자열로 일괄 채워 넣습니다.
# -> 머신러닝 모델이 결측치 에러를 내지 않도록 막고, 'NA'를 하나의 독립된 범주로 활용해 데이터 손실을 방지하기 위해 사용합니다.

numerical_cols = df.select_dtypes(include=['number']).columns
# .select_dtypes(include=['number'])는 정수와 실수 등 모든 숫자형 컬럼을 골라내고, .columns는 이름표를 추출합니다.
# -> 수치 연산이 필요한 숫자 데이터 영역만 별도로 정확하게 분리하기 위해 사용합니다.

df[numerical_cols] = df[numerical_cols].fillna(0.0)
# .fillna(0.0)은 숫자형 컬럼들의 비어 있는 빈칸(NaN)을 실수형 0.0으로 메워 넣습니다.
# -> 숫자가 비어 있으면 사칙연산이나 행렬 곱이 불가능하므로, 가장 중립적인 값인 0으로 대체하여 정상적인 연산이 가능하도록 만들기 위해 사용합니다.

## Q1 — Mode of `industry`

What is the most frequent observation (mode) for the column `industry`?

- `NA`
- `technology`
- `healthcare`
- `retail`


In [10]:
q1_mode = df['industry'].value_counts().index[0]
# 1) industry 컬럼의 값별 등장 횟수를 내림차순 정렬하고(value_counts), 
# 2) 가장 많이 등장한 순서 중 첫 번째 항목의 이름(.index[0])을 가져와 가장 빈도가 높은 최빈값을 구합니다.

print(f"Q1 최빈값: {q1_mode}")
# 구한 최빈값 결과를 화면에 출력하여 과제 제출 페이지의 보기와 대조할 수 있도록 합니다.

Q1 최빈값: technology


## Q2 — Correlation matrix (numerical features)

What are the two features that have the biggest correlation?

- `interaction_count` and `lead_score`
- `number_of_courses_viewed` and `lead_score`
- `number_of_courses_viewed` and `interaction_count`
- `annual_income` and `interaction_count`

Only consider the pairs above.


In [12]:
numerical_df = df.select_dtypes(include=['number'])
# .select_dtypes(include=['number'])는 데이터프레임에서 정수와 실수 형태의 숫자형 컬럼들만 골라냅니다.
# -> 상관계수는 문자가 아닌 숫자 데이터끼리만 수학적 계산이 가능하므로, 수치형 데이터셋만 따로 분리하기 위해 사용합니다.

corr_matrix = numerical_df.corr()
# .corr()은 피처들 간의 피어슨 상관계수를 계산하여 전체 행렬(표) 형태로 반환합니다.
# -> 수치형 변수들 상호 간에 양의 상관관계가 있는지, 음의 상관관계가 있는지 전체적인 관계를 한눈에 파악하기 위해 사용합니다.

print("1. interaction_count & lead_score:", corr_matrix.loc['interaction_count', 'lead_score'])
print("2. number_of_courses_viewed & lead_score:", corr_matrix.loc['number_of_courses_viewed', 'lead_score'])
print("3. number_of_courses_viewed & interaction_count:", corr_matrix.loc['number_of_courses_viewed', 'interaction_count'])
print("4. annual_income & interaction_count:", corr_matrix.loc['annual_income', 'interaction_count'])
# .loc[행, 열]은 상관관계 행렬에서 특정 변수 쌍의 교차 지점에 있는 상관계수 값을 정확하게 콕 집어 추출합니다.
# -> 과제 보기에서 제시한 4가지 변수 쌍의 상관계수 값을 각각 확인하고 가장 큰 값을 고르기 위해 사용합니다

1. interaction_count & lead_score: 0.9287123432397134
2. number_of_courses_viewed & lead_score: 0.769352062184664
3. number_of_courses_viewed & interaction_count: 0.7216090038937035
4. annual_income & interaction_count: 0.10637458469219016


## Split the data

Use these exact calls. `converted` must not stay in the feature dataframes.


In [15]:
target_col = "converted"

df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
# train_test_split()을 사용하여 전체 데이터를 학습/검증용(80%)과 최종 테스트용(20%)으로 무작위 분할합니다.
# -> random_state=42를 지정하여 난수 생성 패턴을 고정하고 결과를 재현 가능하게 만들기 위해 사용합니다.

df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=42
)
# 앞서 분할한 학습/검증용 데이터에서 25%(전체 기준 20%)를 검증용으로 떼어내어, 최종 6:2:2 (Train:Val:Test) 비율을 완성합니다.
# -> 모델 학습과 성능 검증을 분리하여 과적합을 방지하고 일반화 성능을 평가하기 위해 사용합니다.

df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)
# .reset_index(drop=True)는 쪼개지면서 뒤섞인 기존 인덱스 번호를 초기화하고 0부터 순차적으로 다시 정렬합니다.
# -> 이후 데이터 인덱싱 작업에서 발생할 수 있는 에러를 방지하고 깔끔한 관리를 위해 사용합니다.

y_train = df_train[target_col].values
y_val = df_val[target_col].values
y_test = df_test[target_col].values
# 데이터프레임에서 정답 컬럼('converted')만 추출한 뒤 .values를 통해 넘파이 배열로 변환합니다.
# -> 사이킷런 모델에 정답 데이터로 곧바로 전달하기 위해 사용합니다.

df_train_features = df_train.drop(columns=[target_col])
df_val_features = df_val.drop(columns=[target_col])
df_test_features = df_test.drop(columns=[target_col])
# .drop(columns=[target_col])을 통해 정답 컬럼을 특성 데이터프레임에서 완전히 제거합니다.
# -> 과제 요구사항대로 모델이 정답을 미리 보고 학습하는 데이터 누수(Data Leakage)를 원천 차단하기 위해 사용합니다.

## Q3 — Mutual information (training set)

Calculate the mutual information score between `converted` and other categorical variables.
Use the training set only. Round with `round(score, 2)`.

Which of these variables has the biggest mutual information score?

- `industry`
- `location`
- `lead_source`
- `employment_status`


In [25]:
cat_features = ["industry", "location", "lead_source", "employment_status"]
# 과제 보기에서 지정한 4개의 범주형 변수 이름(industry, location, lead_source, employment_status)만 골라서 리스트 형태로 직접 정의합니다.
# -> 불필요한 다른 컬럼들을 배제하고 오직 과제에서 요구하는 보기들만 정확하게 타겟팅하기 위해 사용합니다.

mi_frame = df_train_features[cat_features].fillna("NA")
# 1) df_train_features[cat_features]는 지정된 보기 컬럼들만 훈련 세트에서 추출하고, 
# 2) .fillna("NA")는 혹시라도 전처리 과정에서 누락되었을 수 있는 빈칸(NaN)을 "NA" 문자열로 안전하게 메워 넣습니다.
# -> mutual_info_score 함수가 결측치 때문에 에러를 발생시키는 상황을 원천 방지하기 위해 사용합니다.

def calculate_mi(series):
    # 전달받은 범주형 시리즈 데이터와 훈련 세트의 정답(y_train) 간의 상호 정보량 점수를 계산하는 커스텀 함수를 선언합니다.
    return mutual_info_score(series, y_train)
    # mutual_info_score()는 두 변수 간에 공유하는 정보량(연관성)을 수치로 계산하여 반환합니다.
    # -> 어떤 범주형 변수가 정답 예측에 가장 큰 단서를 제공하는지 정량적으로 측정하기 위해 사용합니다.

mi_scores = mi_frame.apply(calculate_mi)
# .apply(calculate_mi)는 지정된 4개의 컬럼 각각에 위에서 정의한 상호 정보량 계산 함수를 일괄 적용합니다.
# -> 데이터 누수를 막기 위해 반드시 오직 훈련 세트(Train set)만을 기준으로 연산을 수행하기 위함입니다.

mi_scores = mi_scores.round(2)
# .round(2)를 통해 계산된 점수들을 소수점 둘째 자리까지 반올림합니다.
# -> 과제 보기의 값들과 깔끔하게 대조하고 가독성을 높이기 위해 사용합니다.

print("--- Q3: 훈련 세트 범주형 변수별 상호 정보량 점수 ---")
print(mi_scores.sort_values(ascending=False))
# .sort_values(ascending=False)를 통해 점수가 높은 순서대로 내림차순 정렬하여 출력합니다.
# -> 제시된 보기 4개 중 어떤 변수의 상호 정보량 점수가 가장 큰지 직관적으로 확인하고 정답을 고르기 위해 사용합니다.

--- Q3: 훈련 세트 범주형 변수별 상호 정보량 점수 ---
lead_source          0.03
employment_status    0.02
location             0.00
industry             0.00
dtype: float64


## Q4 — Logistic regression accuracy

Include categorical variables with one-hot encoding.
Fit with:

`LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)`

Accuracy on validation, rounded to 2 decimal digits.

- 0.55
- 0.65
- 0.75
- 0.85


In [28]:
from sklearn.feature_extraction import DictVectorizer
# 범주형(문자열) 데이터를 컴퓨터가 이해할 수 있는 0과 1의 형태(원-핫 인코딩)로 변환해 주는 DictVectorizer 클래스를 임포트합니다.
# -> 머신러닝 모델이 글자 데이터를 입력받아 연산할 수 있도록 변환하기 위해 사용합니다.

def fill_features(frame):
    # 전달받은 데이터프레임의 복사본을 만들어 결측치(NaN)를 안전하게 채워 넣는 커스텀 함수를 선언합니다.
    out = frame.copy()
    cat_cols = out.select_dtypes(include=["object", "string"]).columns
    num_cols = out.select_dtypes(include="number").columns
    out[cat_cols] = out[cat_cols].fillna("NA")
    # .fillna("NA")를 통해 범주형 컬럼들의 비어 있는 빈칸을 "NA" 문자열로 메웁니다.
    out[num_cols] = out[num_cols].fillna(0.0)
    # .fillna(0.0)을 통해 수치형 컬럼들의 비어 있는 빈칸을 실수형 0.0으로 메웁니다.
    # -> 모델이 결측치(NaN) 에러를 일으키는 것을 원천 방지하기 위해 사용합니다.
    return out

train_f = fill_features(df_train_features)
val_f = fill_features(df_val_features)
# 정의한 fill_features 함수를 훈련 세트와 검증 세트의 특성 데이터프레임에 각각 적용하여 빈칸을 완전히 정리합니다.
# -> 벡터화 과정 및 로지스틱 회귀 모델 학습 시 발생할 수 있는 결측치 에러를 방지하기 위해 사용합니다.

train_dicts = train_f.to_dict(orient='records')
val_dicts = val_f.to_dict(orient='records')
# 1) .to_dict(orient='records')는 데이터프레임의 각 행을 파이썬 딕셔너리 형태로 변환하여 리스트로 묶어줍니다.
# -> DictVectorizer가 인식할 수 있는 입력 데이터 구조로 형태를 맞추기 위해 사용합니다.

dv = DictVectorizer(sparse=False)
# DictVectorizer(sparse=False)는 결과를 메모리 효율적인 희소 행렬 대신 일반 2차원 넘파이 배열 형태로 반환하도록 객체를 생성합니다.
# -> 이후 모델에 데이터를 곧바로 전달하기 편리하게 만들기 위해 사용합니다.

X_train = dv.fit_transform(train_dicts)
X_val = dv.transform(val_dicts)
# 1) .fit_transform()은 훈련 데이터를 이용해 인코딩 기준(어떤 카테고리가 컬럼이 될지)을 학습하고 곧바로 변환하며, 
# 2) .transform()은 검증 데이터를 앞서 학습된 기준에 맞춰 변환만 수행합니다.
# -> 검증 세트의 정보가 훈련 과정에 새어나가는 데이터 누수를 완벽하게 차단하기 위해 사용합니다.

model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
# 지정된 파라미터(solver='liblinear', C=1.0, max_iter=1000, random_state=42)를 적용한 로지스틱 회귀 모델 객체를 생성합니다.
# -> 이진 분류(가입 여부 예측) 문제를 풀기 위한 학습 알고리즘을 준비하기 위해 사용합니다.

model.fit(X_train, y_train)
# .fit(X_train, y_train)은 인코딩된 훈련 특성 행렬(X_train)과 정답 레이블(y_train)을 입력받아 모델을 학습시킵니다.
# -> 데이터 속의 패턴과 가중치를 찾아내어 예측 능력을 갖추게 만들기 위해 사용합니다.

y_val_pred = model.predict(X_val)
# .predict(X_val)은 학습된 모델에 검증 특성 데이터(X_val)를 입력하여 예측값(0 또는 1)을 생성합니다.
# -> 모델이 한 번도 보지 못한 검증 세트를 대상으로 실력을 평가하기 위해 사용합니다.

val_accuracy = accuracy_score(y_val, y_val_pred)
# accuracy_score()는 모델이 예측한 값(y_val_pred)과 실제 정답(y_val)을 비교하여 정확한 비율을 계산합니다.
# -> 모델의 전체적인 분류 성능을 객관적인 수치로 측정하기 위해 사용합니다.

print(f"Q4 검증 정확도: {round(val_accuracy, 2)}")
# 계산된 정확도 점수를 소수점 둘째 자리까지 반올림하여 화면에 출력합니다.
# -> 과제 제출 페이지의 보기와 대조하기 위해 사용합니다.

Q4 검증 정확도: 0.65


## Q5 — Feature elimination

Train with the same features/parameters as Q4 (without rounding).
Exclude each feature, retrain, record accuracy, compute difference vs original.

Which of the following has the smallest difference?

- `'lead_source'`
- `'number_of_courses_viewed'`
- `'interaction_count'`

The difference does not have to be positive.


In [30]:
features_to_test = ["lead_source", "number_of_courses_viewed", "interaction_count"]
# 과제 원문에서 지정한 3개의 테스트 피처 이름 리스트를 정의합니다.
# -> 특정 피처를 하나씩 차례대로 제외하면서 모델 성능에 미치는 영향을 체계적으로 테스트하기 위해 사용합니다.

baseline_acc = val_accuracy
# 반올림하지 않은 원본 모델의 검증 정확도(val_accuracy)를 기준값(baseline_acc) 변수에 고스란히 담아둡니다.
# -> 피처를 하나씩 뺐을 때 원래 실력에 비해 정확도가 얼마나 미세하게 변동되는지 정밀하게 비교하기 위해 사용합니다

print(f"기준(Baseline) 정확도 원점수: {baseline_acc}")
# 비교의 기준이 되는 원본 정확도 수치를 화면에 출력하여 눈으로 확인합니다.
# -> 실험 시작 전 기준점을 명확히 인지하기 위해 사용합니다.

def drop_and_fill(frame, col_to_drop):
    # 데이터프레임과 제외할 컬럼 이름을 입력받아, 해당 컬럼을 도려내고 결측치를 메운 새로운 데이터프레임을 반환하는 함수를 정의합니다.
    # -> 반복문 속에서 각 피처를 안전하고 일관되게 제거하기 위해 사용합니다.
    
    out = frame.drop(columns=[col_to_drop], errors='ignore').copy()
    # 1) .drop(columns=[col_to_drop], errors='ignore')은 데이터프레임에서 현재 대상 컬럼을 완전히 제거하며, 
    # 2) .copy()는 원본 데이터프레임에 영향을 주지 않도록 독립된 메모리 복사본을 만듭니다.
    # -> 특정 변수가 빠진 새로운 특성 셋을 오염 없이 구성하기 위해 사용합니다.
    
    cat_cols = out.select_dtypes(include=["object", "string"]).columns
    num_cols = out.select_dtypes(include="number").columns
    # 데이터프레임 내부에서 범주형(문자열) 컬럼과 수치형 컬럼의 이름표 목록을 각각 추출합니다.
    # -> 데이터 타입별로 알맞은 결측치 채우기 방식을 적용하기 위해 사용합니다.
    
    out[cat_cols] = out[cat_cols].fillna("NA")
    out[num_cols] = out[num_cols].fillna(0.0)
    # 1) .fillna("NA")는 문자열 빈칸을 "NA"로 메우고, 
    # 2) .fillna(0.0)은 숫자 빈칸을 0.0으로 메웁니다.
    # -> 모델이 결측치(NaN) 때문에 학습 중 에러를 일으키는 상황을 원천 차단하기 위해 사용합니다.
    
    return out

기준(Baseline) 정확도 원점수: 0.645


In [31]:
for col in features_to_test:
    # 3개의 피처 이름을 담은 리스트에서 하나씩 요소를 꺼내어 반복문을 실행합니다.
    # -> 3개의 피처를 각각 하나씩 제외하는 실험을 반복 자동화하기 위해 사용합니다.

    # 1. 훈련 및 검증 데이터셋에 특정 피처 제거 전처리 적용
    t_f = drop_and_fill(df_train_features, col)
    v_f = drop_and_fill(df_val_features, col)
    # 앞서 정의한 drop_and_fill 함수를 이용해 현재 타겟 피처가 빠진 훈련/검증 특성 셋을 각각 생성합니다.

    # 2. 딕셔너리 변환 및 DictVectorizer 원-핫 인코딩 수행
    t_dicts = t_f.to_dict(orient="records")
    v_dicts = v_f.to_dict(orient="records")
    # .to_dict(orient="records")는 데이터프레임의 행들을 딕셔너리 리스트 형태로 변환합니다.
    
    dv_q5 = DictVectorizer(sparse=False)
    X_t = dv_q5.fit_transform(t_dicts)
    X_v = dv_q5.transform(v_dicts)
    # 1) .fit_transform()은 훈련 세트로 인코딩 기준을 학습하고 변환하며, 2) .transform()은 검증 세트를 그 기준에 맞추어 변환합니다.
    # -> 데이터 누수를 완벽하게 차단하고 공정한 인코딩 환경을 만들기 위해 사용합니다.

    # 3. 로지스틱 회귀 모델 학습 및 예측
    m_q5 = LogisticRegression(solver="liblinear", C=1.0, max_iter=1000, random_state=42)
    m_q5.fit(X_t, y_train)
    # 지정된 파라미터로 모델을 생성하고, 특성이 하나 빠진 훈련 행렬(X_t)과 정답(y_train)으로 모델을 학습시킵니다.
    
    pred_q5 = m_q5.predict(X_v)
    acc_q5 = accuracy_score(y_val, pred_q5)
    # .predict()로 검증 예측값을 뽑고, accuracy_score()로 정확도를 측정합니다.

    # 4. 기준 정확도와의 차이(signed difference) 계산 및 출력
    diff = baseline_acc - acc_q5
    # 원점수(baseline_acc)에서 피처를 제외한 뒤의 점수(acc_q5)를 뺍니다.
    # -> 과제에서 요구하는 정확도 차이(Difference)를 부호 포함하여 정확히 산출하기 위해 사용합니다.

    print(f"[{col} 제외 시] 정확도: {acc_q5:.5f} | 차이 (기준 - 제외): {diff:.5f}")
    # 결과값을 소수점 아래 정밀하게 출력하여 어떤 피처의 차이가 가장 작은지 눈으로 확인할 수 있게 합니다.

[lead_source 제외 시] 정확도: 0.64200 | 차이 (기준 - 제외): 0.00300
[number_of_courses_viewed 제외 시] 정확도: 0.64300 | 차이 (기준 - 제외): 0.00200
[interaction_count 제외 시] 정확도: 0.60100 | 차이 (기준 - 제외): 0.04400


## Q6 — Regularized logistic regression

Try `C` in `[0.000001, 0.00001, 0.0001, 0.001]`.
Accuracy on validation, rounded to 3 decimal digits.
If multiple options, select the smallest `C`.

- 0.000001
- 0.00001
- 0.0001
- 0.001


In [7]:
C_values = [0.000001, 0.00001, 0.0001, 0.001]


for c_val in C_values:
    m = LogisticRegression(solver="liblinear", c_val, max_iter=1000, random_state=42)
    m.fit(X_train, y_train)
    acc = accuracy_score(y_val, m.predict(X_val))
    print(f"C={c_val} 정확도={acc:.5f} round3={round(acc, 3)}")
    
    